# MD simulations in the NVT ensamble using a Nosé-Hoover Chain thermostat

## Moldule imports

In [ ]:
from ase import io, units
from ase.constraints import FixAtoms, FixCom, FixSubsetCom
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.md.nose_hoover_chain import NoseHooverChainNVT

import pfp_api_client
from pfp_api_client.pfp.calculators.ase_calculator import ASECalculator
from pfp_api_client.pfp.estimator import Estimator, EstimatorCalcMode

estimator = Estimator(calc_mode=EstimatorCalcMode.PBE_PLUS_D3, model_version='v9.0.0')
calculator = ASECalculator(estimator)

print(f'pfp_api_client: {pfp_api_client.__version__}')

## Parameters

In [ ]:
input_file = '../CPO-27-Co_PFOA_relaxed.traj@0'
output_file = 'CPO-27-Co_PFOA_MD.traj'
trajectory_file = 'CPO-27-Co_PFOA_MD-trajectory.traj' # None to skip
log_file = '-' # '-' for stdout

temperature = 400.0 # K
timestep = 1.0 # fs
tdamp = 100.0 # fs, the characteristic time scale for the thermostat. Typically, it is set to 100 times of timestep.

md_steps = 1000000
log_steps = 500

### Parsing of parameters

In [ ]:
temperature = float(temperature)
timestep = float(timestep)
tdamp = float(tdamp)

md_steps = int(md_steps)
log_steps = int(log_steps)

In [ ]:
atoms = io.read(input_file, index=-1)
atoms.calc = calculator
# atoms.set_constraint(FixCom())

MaxwellBoltzmannDistribution(atoms, temperature_K=temperature)
Stationary(atoms)

dyn = NoseHooverChainNVT(atoms,
                         timestep=timestep * units.fs,
                         temperature_K=temperature,
                         tdamp=tdamp * units.fs,
                         trajectory=trajectory_file,
                         logfile=log_file, loginterval=log_steps)

status = dyn.run(md_steps)
io.write(output_file, atoms)